## House Price Regression

### Introducing the Problem/Dataset

This project aims to explore the vastly different kinds of variables involved, whether seemingly central or ancillary details, that  may or may not contribute to sales prices for houses. A home buyer heading into price negotiations is likely to consider basic features of the property such as number of bedrooms or having a white picket fence. It is exceedingly difficult particularly for the inexperienced customer (typically one only purchases a single property), to properly consider and factor in all of the contextual information surrounding their property of interest. It is easy to miss significant features that seem irrelevant at a cursory glance such as height of the basement or the proximity to an east-west railroad. We examine an Ames Housing dataset compiled by Dean De Cock. It contains a whopping 79 explanatory variables meticulously describing the aspects of residental homes in Ames, Iowa. Given this context, we aim to predict the final price of each home, as requested by the Kaggle competition associated with this dataset.

In [ ]:
import numpy as np 
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import LassoCV
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, PolynomialFeatures, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.metrics import classification_report

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

### What is regression and how does it work?

In the context of machine learning, regression is a supervised machine learning technique used to predict continuous target values through modeling the relationship between input variables and a dependent variable. There are many types of regression that we have mentioned such as linear regression, polynomial regression, stepwise regression, ridge regression, lasso regression, and ElasticNet regression, etc. Linear regression specifically, aims to fit linear model with coefficients in the $w$ vector, to minimize the so-called objective function, which is defined as the residual sum of squares between the observed targets in the dataset and the targets as predicted by the linear approximation.

Two different ways to train a linear regression model were discussed:
1. Utilizing a "closed-form" equation to directly compute the model parameters that best fit the model to the training set, such that the those parameters minimize the objective function over the training set. This is the technique used by the LinearRegression class available through the Scikit-Learn library. which employs a ordinary least squares estimator for linear regression. Within the course, the "closed-form" equation given, is for the case of simple linear regression which is just a basic case of linear regression which involves only a single explanatory variable.

In the case of simple linear regression, we wish to find the parameters minimizing the residual sum of squares in the following equation:

$$y = b + mx$$

such that 

$$\hat{b}= \bar{y} - \hat{m}\bar{x}$$

and we may derive that

$$\hat{m} = \frac{\sum_{i=1}^{n}{(x_i-\bar{x})(y_i-\bar{y})}}{\sum_{i=1}^{n}{(x_i-\bar{x})^2}}$$

Derivation:

The residual sum of squares representing the error can be expressed as follows.

$$ SSE(b, m) = \sum_{i=1}^{n}{(y_i - \hat{y}_i)^2} = \sum_{i=1}^{n}{((y_i - b - mx_i)^2} $$

Compute partial derivatives:

$$ \frac{\partial SSE}{\partial b} = \frac{\partial}{\partial b}(\sum_{i=1}^{n}{(y_i-b-mx_i)^2}) $$
$$ = \sum_{i=1}^{n}{2(y_i - b - mx_i)(-1)} $$
$$ = -2\sum_{i=1}^{n}{(y_i-b-mx_i)} $$

$$ \frac{\partial SSE}{\partial m} = \frac{\partial}{\partial m}(\sum_{i=1}^{n}{(y_i-b-mx_i)^2})$$
$$ = \sum_{i=1}^{n}{2(y_i-b-mx_i)(-x_i)} $$
$$ = -2 \sum_{i=1}^{n}{(x_iy_i-bx_i-mx_i^2)} $$

Setting both derivatives to zero yields the following equations:

\begin{equation}
\sum_{i=1}^{n}{y_i} = \hat{b}n+\hat{m}\sum_{i=1}^{n}{x_i}
\tag{1}
\end{equation}

\begin{equation}
\sum_{i=1}^{n}{x_iy_i} = \hat{b}\sum_{i=1}^{n}{x_i} + \hat{m}\sum_{i=1}^{n}{x_i^2}
\tag{2}
\end{equation}

from (1), we obtain the expression for the bias weight:
$$\frac{1}{n}\sum_{i=1}^{n}{y_i} = \hat{b} + \hat{m} \cdot \frac{1}{n} \sum_{i=1}^{n}{x_i}$$

$$ \bar{y_i} = \hat{b} + \hat{m}\bar{x} $$
$$ \hat{b} = \bar{y} - \hat{m}\bar{x} $$

from (2), we obtain the expression for the $m$

$$ \sum_{i=1}^{n}{x_iy_i} = (\bar{y} - \hat{m}\bar{x})\sum_{i=1}^{n}{x_i} + \hat{m}\sum_{i=1}^{n}{x_i^2}$$

$$ \sum_{i=1}^{n}{x_iy_i} - \bar{y}\sum_{i=1}^{n}{x_i} = \hat{m}(\sum_{i=1}^{n}{x_i^2}-\bar{x}\sum_{i=1}^{n}{x_i})$$

$$ \hat{m} = \frac{\sum_{i=1}^{n}{x_iy_i-\bar{y}\sum_{i=1}^{n}{x_i}}}{\sum_{i=1}^{n}{x_i^2}-\bar{x}\sum_{i=1}^{n}{x_i}} \tag{3}$$

The numerator from (3) can be rewritten as follows:

$$ \sum_{i=1}^{n}{x_iy_i-\bar{y}\sum_{i=1}^{n}{x_i}} $$
$$ = \sum_{i=1}^{n}{x_iy_i-n\bar{x}\bar{y}}$$
$$ = \sum_{i=1}^{n}{x_iy_i-n\bar{x}\bar{y}-n\bar{x}\bar{y}+n\bar{x}\bar{y}}$$
$$ = \sum_{i=1}^{n}{x_iy_i}-\bar{y}\sum_{i=1}^{n}{x_i}-\bar{x}\sum_{i=1}^{n}{y_i}+\sum_{i=1}^{n}{\bar{x}\bar{y}}$$
$$ = \sum_{i=1}^{n}{(x_iy_i-x_i\bar{y}-\bar{x}y_i+\bar{x}\bar{y})}$$
$$ = \sum_{i=1}^{n}{(x_i-\bar{x})(y_i-\bar{y})} \tag{4}$$

The denominator from (3) can be rewritten as follows:

$$\sum_{i=1}^{n}{x_i^2}-\bar{x}\sum_{i=1}^{n}{x_i}$$
$$= \sum_{i=1}^{n}{x_i^2}-n\bar{x}^2$$
$$= \sum_{i=1}^{n}{x_i^2}-2n\bar{x}\bar{x}+n\bar{x}^2$$
$$= \sum_{i=1}^{n}{x_i^2}-2\bar{x}\sum_{i=1}^{n}{x_i}+\sum_{i=1}^{n}{\bar{x}^2}$$
$$= \sum_{i=1}^{n}{(x_i^2-2\bar{x}x_i+\bar{x}^2)}$$
$$=\sum_{i=1}^{n}{(x_i-\bar{x})^2} \tag{5}$$

Combining our results from (4) and (5), we obtain our original expression for $\hat{m}$.

$$\hat{m} = \frac{\sum_{i=1}^{n}{(x_i-\bar{x})(y_i-\bar{y})}}{\sum_{i=1}^{n}{(x_i-\bar{x})^2}}$$

The closed-form solution used by the ordinary least squares estimator implemented in the LinearRegression class is given by the following normal equation:

$$\hat{\vec{w}} = (\mathbf{X}^T\mathbf{X})^{-1}\mathbf{X}^T\vec{y}$$

where $\hat{\vec{w}}$ is the value of $\vec{w}$ that minimizes the objective function, $\vec{y}$ is the vector of target values from $y^{(1)}$ to $y^{(n)}$, and $\mathbf{X}$ is the design matrix, composed of feature column vectors with column $x_0$ always equal to 1.

2. Utilizing an iterative optimization approach known as gradient descent which gradually tweaks the model parameters to minimize the objective function over the training set, which should eventually converge to the same result as the first method. One important hyperparameter in gradient descent is called the learning rate. The basic idea behind learning rate is to control how quickly the model updates the parameters during the training process. If it is too small, it means slow convergence and it may not be able to escape an undesirable local minimum, and if it is too big, it can lead to oscillating behavior such that it may not even converge. There are several variants of gradient descent including batch GD, mini-batch GD, and stochastic GD. Concisely, with batch gradient descent, we use all the training examples, with mini-batch, we use a constant number of training examples, with stochastic gradient descent, we use one training example and update after each.

A summary of the algorithm is as follows:
- Set the learning rate $\eta$ to some small value.
- Set with some initial guess for $\vec{w}^0$ and set $\tau = 0$
- Loop for some number of epochs E or until the J no longer improves
- Within the loop, increment $\tau$ and update the vector of weights

The general gradient descent equation is as follows:
$$\vec{w}^{\tau+1}=\vec{w}^{\tau}-\eta\nabla{J}(\vec{w}^{\tau})$$

The negative of $\nabla{J}$ (gradient of J, the objective function) can be interpreted as the direction of steepest decrease. Remember that the goal to minimize the objective function. $\eta$ represents the learning rate. We can expand the above update equation for batch gradient descent using the mean sum of squares function as the objective, where $h_{\vec{w}}(\vec{x}^{(n)}) = \vec{w}^T\vec{x}^{(n)}$ is our hypothesis function which yields predictions:

$$\vec{w}^{\tau+1}=\vec{w}^{\tau}-\eta\frac{1}{N}\sum_{n=1}^{N}{(h_{\vec{w}}(\vec{x}^{(n)}) - y_n)\vec{x}^{(n)}}$$

There are also other adaptive learning rate methods which could be understood as variations of vanilla gradient descent, such as momentum, Adagrad, Adadela, RMSProp, and Adam.

### Experiment 1: Data understanding

Let's inspect the shape and type of feature columns.

In [ ]:
ames_file_path = '/kaggle/input/house-prices-advanced-regression-techniques/train.csv'
ames_data = pd.read_csv(ames_file_path)
print(f"Full training dataset shape: {ames_data.shape}")
print(ames_data.info())

Let's print the first 5 entries of our dataset instead of all 1460, and observe some sample values from all 81 dimensions of our dataset

In [ ]:
display(ames_data.head())

Let's observe the statistics of the features. We can see that different features have vastly different scales of mean and standard deviation.

In [ ]:
display(ames_data.describe())

Let's observe the distribution. It appears that the distribution of sale values peaks at around 150000, with very few values exceeding 400000 at the right tail end of the distribution.

In [ ]:
plt.figure(figsize=(10, 10))
sns.histplot(ames_data['SalePrice'], bins=100, kde=True);

Let's investigate the numeric types. There are several strong correlations between the numerical features that stand out including more obvious ones, such as those between SalePrice and OverallQual, or TotalRoomsAboveGround and GrLivArea, or GarageArea and GarageCars.

In [ ]:
df_names_num = ames_data.select_dtypes(include = ['float64', 'int64'])
corr_matrix = df_names_num.corr()
significant_num_features = corr_matrix['SalePrice'].abs().sort_values(ascending=False).head(10).index

ax = sns.heatmap(df_names_num[significant_num_features].corr(), fmt='.2f', annot=True, cmap=sns.cubehelix_palette(as_cmap=True))
plt.title("Select numerical feature correlations including SalePrice")
ax.xaxis.tick_bottom()

### Experiment 1: Pre-processing

Check null value status.

In [ ]:
missing = ames_data.isnull().sum().sort_values(ascending=False)
missing[missing > 0]

In [ ]:
X = ames_data.drop(['Id', 'SalePrice'], axis=1)

cat_cols = X.select_dtypes(include=["object"]).columns
num_cols = [c for c in X.columns if c not in cat_cols]
X[cat_cols] = X[cat_cols].fillna('Missing')

numeric_preprocess = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

cat_preprocess = Pipeline(steps=[
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_preprocess, num_cols),
        ("cat", cat_preprocess, cat_cols),
    ],
)

y = ames_data.SalePrice

### Experiment 1: Modeling

In [ ]:
#features = ['MSSubClass', 'LotArea', 'OverallQual', 'OverallCond', 'YearBuilt', 'YearBuilt', 'YearRemodAdd', 'MasVnrArea']
#X_1 = ames_data[features]
linear_model = Pipeline(steps=[
    ("preprocess", preprocess),
    ("regressor", LinearRegression())
])

linear_model.fit(X, y)

### Experiment 1: Evaluation

In [ ]:
test_data_path = '/kaggle/input/house-prices-advanced-regression-techniques/test.csv'
test_data = pd.read_csv(test_data_path)
test_preds = linear_model.predict(test_data)

#### Creating submission.csv

In [ ]:
output = pd.DataFrame({'Id': test_data.Id,
                      'SalePrice': test_preds})
output.to_csv('submission.csv', index=False)

Repeating the above by manually splitting the data instead of submitting to competition

In [ ]:
X = ames_data.drop(['Id', 'SalePrice'], axis=1)
y = ames_data['SalePrice']

num_features = X.select_dtypes(include=['int64', 'float64']).columns
cat_features = X.select_dtypes(include=['object']).columns

X[num_features] = X[num_features].fillna(X[num_features].median())
X[cat_features] = X[cat_features].fillna('Missing')

X_trn, X_tst, y_trn, y_tst = train_test_split(X, y, test_size=0.2, random_state=10)

preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_features),
        ("cat", OneHotEncoder(handle_unknown='ignore'), cat_features),
    ],
)

linear_model = Pipeline(steps=[
    ("preprocess", preprocess),
    ("regressor", LinearRegression())
])

linear_model.fit(X_trn, y_trn);
y_hat_tst = linear_model.predict(X_tst)

rmse = np.sqrt(mean_squared_error(y_tst, y_hat_tst))
r2 = r2_score(y_tst, y_hat_tst)

print(f"RMSE: {rmse: .2f}, R^2: {r2:.2f}")

#output2 = pd.DataFrame({'Id': test_data.Id,
#                      'SalePrice': linear_model.predict()})
#output2.to_csv('submission.csv', index=False)

### Experiment 2:

Here I am experimenting by trying subset of different features. It confirms that I must be careful when choosing features to select because I actually got worse results.

In [ ]:
features = ['MSSubClass', 'LotArea', 'OverallQual', 'OverallCond', 'YearBuilt', 'YearRemodAdd', 'MasVnrArea']

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_preprocess, features),
    ],
)

linear_model = Pipeline(steps=[
    ("preprocess", preprocess),
    ("regressor", LinearRegression())
])

linear_model.fit(X_trn[features], y_trn);
y_hat_tst = linear_model.predict(X_tst[features])

rmse = np.sqrt(mean_squared_error(y_tst, y_hat_tst))
r2 = r2_score(y_tst, y_hat_tst)

print(f"RMSE: {rmse: .2f}, R^2: {r2:.2f}")

### Experiment 3:

Using LassoCV, where it seems to perform slightly better than the normal LinearRegression class from Experiment 1.

In [ ]:
lasso_model = Pipeline(steps=[
    ("preprocess", preprocess),
    ("regressor", LassoCV(cv=5, max_iter=50000, random_state=10))
])

lasso_model.fit(X_trn, y_trn);
y_hat_tst = lasso_model.predict(X_tst)

rmse = np.sqrt(mean_squared_error(y_tst, y_hat_tst))
r2 = r2_score(y_tst, y_hat_tst)

print(f"RMSE: {rmse: .2f}, R^2: {r2:.2f}")

### Impact

Based on this problem and data, concisely put, the positive impact would be the reduction of real estate information asymmetry in the housing market through providing data-driven insights to underserved communities. A possible negative impact would amplify inequalities that are already embedded within the training data. As we have learned before, data quality is integral to the success of a good model, much more significantly than model architectural design. Garbage in, garbage out.

### Conclusion

Through the completion of this project, I have learned to apply regression algorithms such as linear regression to train models which predict a continuous dependent variable, which in this case are house prices. I learned that feature selection can improve performance since irrelevant features really just adds noise which presents itself as an obstacle for the model to navigate during the training process. I also learned that trying different variations of regression models can at least marginally improve performance, for instance the basic idea behind classes like Lasso, is adding a regularization term that "encourages" the model to learn small values for the weights which also achieve the goal of error function minimization, in order to mitigate overfitting.

### References

https://seaborn.pydata.org/generated/seaborn.heatmap.html